In [ ]:
# mediapose 

# 0. nose
# 1. left_eye_inner
# 2. left_eye
# 3. left_eye_outer
# 4. right_eye_inner
# 5. right_eye
# 6. right_eye_outer
# 7. left_ear
# 8. right_ear
# 9. mouth_left
# 10. mouth_right
# 11. left_shoulder
# 12. right_shoulder
# 13. left_elbow
# 14. right_elbow
# 15. left_wrist
# 16. right_wrist
# 17. left_pinky
# 18. right_pinky
# 19. left_index
# 20. right_index
# 21. left_thumb
# 22. right_thumb
# 23. left_hip
# 24. right_hip
# 25. left_knee
# 26. right_knee
# 27. left_ankle
# 28. right_ankle
# 29. left_heel
# 30. right_heel
# 31. left_foot_index
# 32. right_foot_index



In [4]:
import cv2
import mediapipe as mp
import numpy as np

def calculate_angle(a, b, c):
    """
    a, b, c: (x, y) 좌표
    b를 기준으로 한 각도 반환 (degree)
    """
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)

    ba = a - b
    bc = c - b

    cosine_angle = np.dot(ba, bc) / (
        np.linalg.norm(ba) * np.linalg.norm(bc)
    )
    angle = np.arccos(np.clip(cosine_angle, -1.0, 1.0))
    return np.degrees(angle)


mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

video_num = "02"
video_path = f"01_Database/{video_num}.mp4"

cap = cv2.VideoCapture(video_path)

KNEE_LANDMARKS = {
    mp_pose.PoseLandmark.LEFT_HIP,
    mp_pose.PoseLandmark.LEFT_KNEE,
    mp_pose.PoseLandmark.LEFT_ANKLE,
}

KNEE_CONNECTIONS = [
    (mp_pose.PoseLandmark.LEFT_HIP, mp_pose.PoseLandmark.LEFT_KNEE),
    (mp_pose.PoseLandmark.LEFT_KNEE, mp_pose.PoseLandmark.LEFT_ANKLE),
]


with mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    smooth_landmarks=True,
    enable_segmentation=False,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7,
) as pose:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        h, w, _ = frame.shape  # ✅ 추가

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = pose.process(frame_rgb)

        if res.pose_landmarks:
            h, w, _ = frame.shape

            hip = res.pose_landmarks.landmark[mp_pose.PoseLandmark.LEFT_HIP.value]
            knee = res.pose_landmarks.landmark[mp_pose.PoseLandmark.LEFT_KNEE.value]
            ankle = res.pose_landmarks.landmark[mp_pose.PoseLandmark.LEFT_ANKLE.value]

            hip_xy = (int(hip.x * w), int(hip.y * h))
            knee_xy = (int(knee.x * w), int(knee.y * h))
            ankle_xy = (int(ankle.x * w), int(ankle.y * h))

            # 🔢 각도 계산
            knee_angle = calculate_angle(hip_xy, knee_xy, ankle_xy)

            # 🔴 점 표시
            cv2.circle(frame, hip_xy, 5, (0, 255, 0), -1)
            cv2.circle(frame, knee_xy, 5, (0, 0, 255), -1)
            cv2.circle(frame, ankle_xy, 5, (0, 255, 0), -1)

            # 🔵 선 표시
            cv2.line(frame, hip_xy, knee_xy, (255, 0, 0), 2)
            cv2.line(frame, knee_xy, ankle_xy, (255, 0, 0), 2)

            # 🧠 각도 텍스트
            cv2.putText(
                frame,
                f"Knee Angle: {int(knee_angle)} deg",
                (knee_xy[0] - 40, knee_xy[1] - 20),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 255),
                2
            )

        cv2.imshow("mediapipe", frame)

        if cv2.waitKey(1) & 0xFF == 27:
            break

cap.release()
cv2.destroyAllWindows()
